# ขั้นที่ 4: Recommendations — สรุปผลและข้อเสนอแนะ

## ภาพรวมของขั้นนี้

รวมผลจากขั้นที่ 1–3 แล้วตอบคำถามว่า **องค์กรควรทำอะไรต่อ**

```
ขั้นที่ 1  Insight         →  รู้ว่าอะไรขับเคลื่อนยอดขาย
ขั้นที่ 2  Forecast        →  มีโมเดลที่แม่นกว่าวิธีเดิม
ขั้นที่ 3  Business impact →  รู้ว่าคุ้มค่าเท่าไร
      ↓
ขั้นที่ 4  Recommendations →  Action, แผน pilot, KPI, roadmap
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True

FIGURES = ROOT / 'reports' / 'figures'
ANALYSIS = ROOT / 'analysis'
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)


def show(fig, name):
    """แสดงกราฟในหน้านี้ และบันทึกสำเนาไว้ใน reports/figures"""
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
from IPython.display import Image, display

---
## 4.1 ผลลัพธ์หลักของโครงการ

In [ ]:
comparison = pd.read_csv(ANALYSIS / 'model_comparison.csv', index_col='method')
impact = pd.read_csv(ANALYSIS / 'business_impact.csv', index_col=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
comparison['WRMSSE'].plot.barh(ax=axes[0], color=['seagreen'] + ['gray'] * (len(comparison) - 1))
axes[0].invert_yaxis()
axes[0].bar_label(axes[0].containers[0], fmt='%.3f')
axes[0].set(title='Forecast accuracy: WRMSSE (lower is better)', ylabel='')

impact.drop('ALL')['per year, 10 stores'].plot.bar(ax=axes[1], rot=0, color='seagreen')
axes[1].bar_label(axes[1].containers[0], fmt='$%,.0f')
axes[1].set(title=f"Annual saving vs baseline, 10 stores (total ${impact.loc['ALL', 'per year, 10 stores']:,.0f})", xlabel='', ylabel='$')
show(fig, '04_1_key_results')

**Insight สำคัญจากขั้นที่ 1**

In [ ]:
for name in ['01_4_snap_effect', '01_4_calendar_effect', '01_5_item_segments']:
    display(Image(FIGURES / f'{name}.png', width=900))

---
## 4.2 Action ที่ได้จาก insight

INSIGHT_ACTIONS_PLACEHOLDER

---
## 4.3 แผนนำไปใช้จริง: Pilot

| หัวข้อ | รายละเอียด |
|---|---|
| **ขอบเขต** | หมวด FOODS ในรัฐ CA: ยอดขายสูง, ได้รับผลจาก SNAP และเป็นสินค้าที่เน่าเสียได้ |
| **วิธีทดลอง** | A/B test: สาขากลุ่มทดลองสั่งของตาม forecast ใหม่ สาขากลุ่มควบคุมใช้วิธีเดิม เป็นเวลา 8–12 สัปดาห์ |
| **เกณฑ์ผ่าน** | Out-of-Stock rate และ waste ลดลงอย่างมีนัยสำคัญ โดยยอดขายไม่ลดลง |

## 4.4 KPI ที่ต้องติดตาม

| กลุ่ม | KPI | ทำไม |
|---|---|---|
| โมเดล | WRMSSE, forecast bias | รู้ว่าโมเดลยังแม่นอยู่หรือเริ่มเพี้ยน |
| การดำเนินงาน | Out-of-Stock rate, fill rate, inventory turnover | ผลต่อหน้าร้านและสต็อก |
| การเงิน | lost sales, markdown %, waste % | ผลเป็นตัวเงินที่ผู้บริหารใช้ตัดสินใจ |

## 4.5 Roadmap

| ระยะ | ช่วงเวลา | งาน |
|---|---|---|
| Pilot | เดือน 1–6 | pipeline ข้อมูล, โมเดล, A/B test FOODS ใน CA |
| Scale | เดือน 6–12 | ขยายทุกหมวดและทุกสาขา เชื่อมกับระบบสั่งซื้ออัตโนมัติ |
| MLOps | ต่อเนื่อง | retrain อัตโนมัติ, ติดตามความแม่นและ drift, ให้ Store Manager ปรับ forecast ได้ |

---
## 4.6 ข้อจำกัดของการศึกษา

- **ข้อมูลมีเพียง 10 สาขา 3 รัฐ** การขยายผลไปทุกสาขาเป็นเพียงการประมาณขนาดของโอกาส
- **ไม่มีข้อมูลสต็อกจริง:** ยอดขาย 0 อาจเกิดจากของหมด ไม่ใช่ไม่มีคนซื้อ (censored demand)
- **ตัวเลขผลกระทบมาจากการจำลอง** ภายใต้สมมติฐานในขั้นที่ 3 ต้องยืนยันด้วย pilot
- **ไม่มีข้อมูลปัจจัยภายนอก** เช่น สภาพอากาศ โปรโมชันของคู่แข่ง และเศรษฐกิจ

## 4.7 แนวทางพัฒนาต่อ

- ทดลองพยากรณ์แบบ recursive (ใช้ lag 1–7 วัน) แล้วรวมกับแบบ direct (ensemble)
- พยากรณ์เป็นช่วง (prediction interval) แทนค่าเดียว เพื่อตั้ง safety stock ได้แม่นขึ้น
- เพิ่มข้อมูลสต็อกจริงเพื่อแยกยอด 0 ที่เกิดจากของหมด

---
## 4.8 สรุป

FINAL_SUMMARY_PLACEHOLDER